# Paso 1 — Terrain Analysis
DEM GLO-30 → pysheds → TWI, flow accumulation, sinks naturales.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.config import load_config, get_bbox
from src.terrain import load_dem, compute_flow
from src.viz import plot_twi_and_sinks, plot_raster
from pathlib import Path

cfg = load_config()
bbox = get_bbox(cfg)
print('BBox Ameghino:', bbox)

## 1.1 Download DEM GLO-30
Via Planetary Computer STAC. Requires: `pip install planetary-computer pystac-client`.

In [ ]:
import pystac_client
import planetary_computer
import rasterio
import numpy as np
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling
import requests
from pathlib import Path

raw_dir = Path('../data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)
dem_out = raw_dir / 'dem_glo30.tif'

if not dem_out.exists():
    catalog = pystac_client.Client.open(
        'https://planetarycomputer.microsoft.com/api/stac/v1',
        modifier=planetary_computer.sign_inplace,
    )
    xmin, ymin, xmax, ymax = bbox
    items = catalog.search(
        collections=['cop-dem-glo-30'],
        bbox=[xmin, ymin, xmax, ymax],
    ).item_collection()
    print(f'Found {len(items)} DEM tiles')

    datasets = [rasterio.open(item.assets['data'].href) for item in items]
    mosaic, mosaic_transform = merge(datasets)
    meta = datasets[0].meta.copy()
    meta.update({'height': mosaic.shape[1], 'width': mosaic.shape[2],
                 'transform': mosaic_transform, 'crs': datasets[0].crs})
    with rasterio.open(dem_out, 'w', **meta) as dst:
        dst.write(mosaic)
    for ds in datasets:
        ds.close()
    print(f'DEM saved: {dem_out}')
else:
    print(f'DEM already exists: {dem_out}')

## 1.2 Terrain analysis (pysheds)

In [ ]:
processed_dir = Path('../data/processed')
processed_dir.mkdir(exist_ok=True)

terrain_outputs = compute_flow(
    dem_path=dem_out,
    output_dir=processed_dir,
    method=cfg['terrain']['flow_direction_method'],
    smooth=cfg['terrain']['smooth_dem'],
    sigma=cfg['terrain']['smooth_sigma'],
)
print('Outputs:', terrain_outputs)

## 1.3 Visualize TWI and sinks

In [ ]:
plot_twi_and_sinks(
    terrain_outputs['twi'],
    terrain_outputs['sinks'],
    output_path='../outputs/twi_sinks.png'
)
plot_raster(terrain_outputs['flow_acc'], title='Flow Accumulation', cmap='Blues',
            output_path='../outputs/flow_acc.png')

## 1.4 Sanity check
Visually verify that known lagunas of Ameghino appear as high-TWI / sinks.
If not → adjust smooth_sigma or switch flow_direction_method in config.yaml.